In [1]:
import torch
from pathlib import Path

In [2]:
# Get checkpoints files from the range 
ckpt_dir = Path("/data/ratna/aug/vflip/eval") # RSG - changed from /data/OM_checkpoints , /data/ratna_retrain/eval
steps = list(range(87500, 137501, 12500))  # RSG - change this for a new range
ckpt_paths = [ ckpt_dir / f"training_{step}" / "teacher_checkpoint.pth" for step in steps]
print(f"Checkpoints to average: {steps}")
print(f"Number of paths: {len(ckpt_paths)}")

Checkpoints to average: [87500, 100000, 112500, 125000, 137500]
Number of paths: 5


In [3]:
# Filter and load checkpoints
state_dicts = [torch.load(str(p), map_location='cpu') for p in ckpt_paths if p.exists()]
print(f"Loaded {len(state_dicts)} checkpoints")

Loaded 5 checkpoints


In [4]:
teacher_dicts = [sd["teacher"] for sd in state_dicts]

In [5]:
print(len(teacher_dicts), teacher_dicts[0])

5 OrderedDict([('backbone.cls_token', tensor([[[-0.0248, -0.0008,  0.0039,  ..., -0.0023, -0.0302, -0.0016]]])), ('backbone.pos_embed', tensor([[[-0.0247, -0.0007,  0.0039,  ..., -0.0023, -0.0302, -0.0016],
         [ 0.1090, -0.0264, -0.0949,  ..., -0.0386, -0.0277,  0.0183],
         [ 0.0064,  0.0066,  0.0219,  ...,  0.0198,  0.0043,  0.0054],
         ...,
         [ 0.0098,  0.0095,  0.0091,  ...,  0.0159,  0.0014,  0.0025],
         [ 0.0064,  0.0069, -0.0052,  ...,  0.0171, -0.0089, -0.0028],
         [ 0.0052,  0.0011, -0.0032,  ...,  0.0086,  0.0035,  0.0013]]])), ('backbone.register_tokens', tensor([[[ 4.6843e-03,  6.7807e-04, -8.1393e-04,  ..., -8.4041e-04,
          -4.5622e-02, -1.2453e-03],
         [-1.6400e-01,  2.9500e-03, -1.7108e-02,  ..., -1.0263e-03,
           3.2499e-02,  2.2867e-04],
         [-9.4838e-03,  5.0312e-05, -1.0583e-01,  ..., -4.2360e-03,
           3.3537e-02,  2.3605e-04],
         [ 2.8505e-02,  6.7222e-04,  4.2526e-03,  ..., -6.1307e-04,
        

In [6]:
averaged_state_dict = {}
for key in teacher_dicts[0].keys():
    averaged_state_dict[key] = sum(td[key] for td in teacher_dicts) / len(teacher_dicts)

In [7]:
# Wrap back in the original structure
output_dict = {'teacher': averaged_state_dict}
print(f"Average dtype: {output_dict['teacher']['backbone.cls_token'].dtype}")

Average dtype: torch.float32


In [9]:
# Save averaged checkpoint
output_dir = Path("/data/ratna/aug/vflip/eval/averaged_87500_to_137500") # RSG - change this for a new range
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "teacher_checkpoint.pth"
torch.save(output_dict, output_path)
print(f"Saved to: {output_path}")

Saved to: /data/ratna/aug/vflip/eval/averaged_87500_to_137500/teacher_checkpoint.pth


The End

In [10]:
ckpt_path = Path("/data/ratna/aug/vflip/eval/averaged_87500_to_137500/teacher_checkpoint.pth")
state_dict = torch.load(ckpt_path, map_location="cpu")
print(state_dict)

{'teacher': {'backbone.cls_token': tensor([[[-0.0233, -0.0008,  0.0041,  ..., -0.0024, -0.0271, -0.0010]]]), 'backbone.pos_embed': tensor([[[-2.3233e-02, -7.4699e-04,  4.1003e-03,  ..., -2.4067e-03,
          -2.7065e-02, -9.8946e-04],
         [ 1.0318e-01, -2.4401e-02, -8.9857e-02,  ..., -3.4467e-02,
          -2.6650e-02,  1.8409e-02],
         [ 3.5816e-03,  8.6385e-03,  1.9759e-02,  ...,  1.7285e-02,
           3.6960e-03,  2.5025e-03],
         ...,
         [ 7.6574e-03,  8.8214e-03,  7.7382e-03,  ...,  1.3380e-02,
          -6.5274e-05,  1.5826e-03],
         [ 4.3363e-03,  5.7227e-03, -1.8036e-03,  ...,  1.1878e-02,
          -6.4164e-03, -2.4652e-03],
         [ 4.0502e-03,  4.9509e-03, -1.6036e-03,  ...,  8.0424e-03,
           2.7062e-03,  2.2307e-03]]]), 'backbone.register_tokens': tensor([[[ 3.7838e-03, -2.5935e-04, -2.0226e-04,  ..., -1.5009e-03,
          -3.9428e-02, -5.2838e-04],
         [-1.5385e-01,  2.7547e-03, -1.5085e-02,  ..., -8.8836e-04,
           2.6711e-02